# DA5107 Integrated Analysis Notebook (English)

This notebook keeps the **key analysis code + results** for each business question in one place.

Scope covered:
1. Expansion pace (business + risk)
2. Risk pricing adequacy and evolution
3. Credit assessment effectiveness
4. Default model performance (bad-customer identification)
5. Data quality assessment
6. Portfolio optimization under 50% capital limit
7. Churn + CLV priority and retention strategy


In [1]:
from pathlib import Path
import csv
import json
from statistics import mean
from IPython.display import display, Markdown
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BASE = Path('5107') if Path('5107').exists() else Path('.')
DATA = BASE / 'data'
FIGURES = BASE / 'figures'

print('Base:', BASE.resolve())
print('Data folder exists:', DATA.exists())
print('Figures folder exists:', FIGURES.exists())


Base: /Users/zhongxinxin/Desktop/nusterm2/DA5107/Assignment Folder/5107
Data folder exists: True
Figures folder exists: True


## 1) Expansion Pace Assessment (Startup Growth vs Risk)

In [2]:
with open(DATA / 'expansion_pace_assessment.json', encoding='utf-8') as f:
    expansion = json.load(f)

verdict = expansion['verdict']
p1 = expansion['phase_metrics']['2012_2015']
p2 = expansion['phase_metrics']['2016_2018']

display(Markdown(f"**Verdict:** {verdict}"))
print('2012-2015 loan count CAGR: {:.2f}%'.format(p1['loan_count_cagr_pct']))
print('2012-2015 loan amount CAGR: {:.2f}%'.format(p1['loan_amt_cagr_pct']))
print('2012-2015 resolved charge-off change: {:+.2f}pp'.format(p1['resolved_co_change_pp']))
print('2016-2018 loan count CAGR: {:.2f}%'.format(p2['loan_count_cagr_pct']))
print('2016-2018 loan amount CAGR: {:.2f}%'.format(p2['loan_amt_cagr_pct']))
print('2016-2018 resolved charge-off change: {:+.2f}pp'.format(p2['resolved_co_change_pp']))
print('Maturity-bias note:', expansion['maturity_bias_note'])


**Verdict:** Initially too fast, then corrected to an adequate pace.

2012-2015 loan count CAGR: 99.03%
2012-2015 loan amount CAGR: 107.36%
2012-2015 resolved charge-off change: +3.99pp
2016-2018 loan count CAGR: 6.76%
2016-2018 loan amount CAGR: 11.33%
2016-2018 resolved charge-off change: -9.58pp
Maturity-bias note: Recent-year default rates are maturity-biased because Current share is high (2018 Current share = 88.32%). Use resolved-loan charge-off rate for fairer trend reading.


In [3]:
rows = []
with open(DATA / 'expansion_yearly_metrics.csv', newline='', encoding='utf-8') as f:
    r = csv.DictReader(f)
    for row in r:
        rows.append({
            'year': int(row['year']),
            'yoy_count': None if row['yoy_loan_count_pct'] in ('', 'None') else float(row['yoy_loan_count_pct']),
            'yoy_amt': None if row['yoy_loan_amt_pct'] in ('', 'None') else float(row['yoy_loan_amt_pct']),
            'resolved_co': float(row['resolved_co_rate_pct'])
        })

years = [x['year'] for x in rows]
count_growth = [x['yoy_count'] for x in rows]
amt_growth = [x['yoy_amt'] for x in rows]
co = [x['resolved_co'] for x in rows]

fig = go.Figure()
fig.add_trace(go.Scatter(x=years, y=count_growth, mode='lines+markers', name='YoY Loan Count Growth (%)'))
fig.add_trace(go.Scatter(x=years, y=amt_growth, mode='lines+markers', name='YoY Loan Amount Growth (%)'))
fig.add_trace(go.Scatter(x=years, y=co, mode='lines+markers', name='Resolved Charge-off Rate (%)'))
fig.update_layout(template='plotly_white', title='Expansion Pace: Growth vs Realized Risk', xaxis_title='Year', yaxis_title='Percent')
fig.show()


## 2) Risk Pricing Adequacy and Time Evolution

In [4]:
def load_csv_dict(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

grade = load_csv_dict(DATA / 'grade_metrics.csv')
monthly = load_csv_dict(DATA / 'monthly_metrics.csv')
bins = load_csv_dict(DATA / 'interest_bin_metrics.csv')

for r in grade:
    r['avg_interest_rate_pct'] = float(r['avg_interest_rate_pct'])
    r['bad_rate_pct'] = float(r['bad_rate_pct'])

for r in monthly:
    r['avg_interest_rate_pct'] = float(r['avg_interest_rate_pct'])

for r in bins:
    r['bad_rate_pct'] = float(r['bad_rate_pct'])

print('Risk grades covered:', [r['grade'] for r in grade])
print('Average pricing spread A->G: {:.2f}pp'.format(grade[-2]['avg_interest_rate_pct'] - grade[0]['avg_interest_rate_pct']))
print('Average bad-rate spread A->G: {:.2f}pp'.format(grade[-2]['bad_rate_pct'] - grade[0]['bad_rate_pct']))


Risk grades covered: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Average pricing spread A->G: 18.36pp
Average bad-rate spread A->G: 33.19pp


In [5]:
x = [r['grade'] for r in grade if r['grade'] in list('ABCDEFG')]
y_rate = [r['avg_interest_rate_pct'] for r in grade if r['grade'] in list('ABCDEFG')]
y_bad = [r['bad_rate_pct'] for r in grade if r['grade'] in list('ABCDEFG')]

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(x=x, y=y_rate, name='Average Interest Rate (%)'), secondary_y=False)
fig.add_trace(go.Scatter(x=x, y=y_bad, mode='lines+markers', name='Bad Loan Rate (%)'), secondary_y=True)
fig.update_layout(template='plotly_white', title='Risk Grade vs Pricing and Credit Risk')
fig.update_yaxes(title_text='Interest Rate (%)', secondary_y=False)
fig.update_yaxes(title_text='Bad Loan Rate (%)', secondary_y=True)
fig.show()


In [6]:
months = [r['month'] for r in monthly]
rates = [r['avg_interest_rate_pct'] for r in monthly]

fig = go.Figure()
fig.add_trace(go.Scatter(x=months, y=rates, mode='lines', name='Monthly Average Rate'))
fig.update_layout(template='plotly_white', title='Interest Rate Trend Over Time', xaxis_title='Issue Month', yaxis_title='Average Interest Rate (%)')
fig.show()


In [7]:
fig = go.Figure()
fig.add_trace(go.Bar(x=[r['interest_bin'] for r in bins], y=[r['bad_rate_pct'] for r in bins], name='Bad Loan Rate (%)'))
fig.update_layout(template='plotly_white', title='Bad Loan Rate by Interest Band', xaxis_title='Interest Band', yaxis_title='Bad Loan Rate (%)')
fig.show()

display(Markdown('**Assessment:** Pricing is broadly risk-aligned (higher-risk grades/bands have higher rates), while high-rate bands also carry higher default risk, so yield quality should be monitored alongside raw margin.'))


**Assessment:** Pricing is broadly risk-aligned (higher-risk grades/bands have higher rates), while high-rate bands also carry higher default risk, so yield quality should be monitored alongside raw margin.

## 3) Credit Assessment Effectiveness (Can it grade risk effectively?)

In [8]:
with open(DATA / 'model_metrics.json', encoding='utf-8') as f:
    mm = json.load(f)

lr = mm['metrics']['logistic_regression']['optimized_for_recall']
rf = mm['metrics']['random_forest']['optimized_for_recall']

print('Logistic (optimized): AUC={:.4f}, Recall_bad={:.4f}, Precision_bad={:.4f}, Threshold={:.4f}'.format(lr['auc'], lr['recall_bad'], lr['precision_bad'], lr['threshold']))
print('RandomForest (optimized): AUC={:.4f}, Recall_bad={:.4f}, Precision_bad={:.4f}, Threshold={:.4f}'.format(rf['auc'], rf['recall_bad'], rf['precision_bad'], rf['threshold']))

better = 'Random Forest' if (rf['recall_bad'], rf['auc']) >= (lr['recall_bad'], lr['auc']) else 'Logistic Regression'
display(Markdown(f'**Current evidence suggests `{better}` is stronger for bad-customer identification under recall-priority policy.**'))


Logistic (optimized): AUC=0.6135, Recall_bad=0.8667, Precision_bad=0.1526, Threshold=0.4391
RandomForest (optimized): AUC=0.7481, Recall_bad=0.6602, Precision_bad=0.2500, Threshold=0.4823


**Current evidence suggests `Logistic Regression` is stronger for bad-customer identification under recall-priority policy.**

In [9]:
models = ['Logistic Regression', 'Random Forest']
auc = [lr['auc'], rf['auc']]
recall = [lr['recall_bad'], rf['recall_bad']]
precision = [lr['precision_bad'], rf['precision_bad']]

fig = go.Figure()
fig.add_trace(go.Bar(name='AUC', x=models, y=auc))
fig.add_trace(go.Bar(name='Recall (Bad)', x=models, y=recall))
fig.add_trace(go.Bar(name='Precision (Bad)', x=models, y=precision))
fig.update_layout(barmode='group', template='plotly_white', title='Default Model Comparison (Recall-Optimized)')
fig.show()


**Management interpretation:**
- If the bank focuses on avoiding bad loans, recall on bad customers is the primary KPI.
- The current model set can rank risk, but threshold governance is necessary to control false positives and business acceptance.
- Recommendation: deploy with monthly backtesting and threshold recalibration by portfolio segment.


## 4) Data Quality Review

In [10]:
with open(DATA / 'data_quality_profile.json', encoding='utf-8') as f:
    dq = json.load(f)

print('Rows:', dq['total_rows'])
print('Columns:', dq['columns'])
print('\nTop missing fields (%):')
for c, p in dq['top_missing_pct'][:10]:
    print(f'- {c}: {p}%')

print('\nKey anomaly counters:')
for k, v in list(dq['anomalies'].items())[:10]:
    print(f'- {k}: {v}')


Rows: 2147635
Columns: 49

Top missing fields (%):
- annual_inc_joint: 94.66%
- mths_since_last_record: 84.11%
- next_pymnt_d: 57.67%
- all_util: 38.33%
- open_acc_6m: 38.32%
- max_bal_bc: 38.32%
- inq_fi: 38.32%
- inq_last_12m: 38.32%
- mths_since_recent_inq: 13.07%
- emp_title: 7.39%

Key anomaly counters:
- total_acc_less_than_open_acc: 2


**Governance direction:**
- Set data standards for mandatory fields at origination.
- Add ingestion checks for schema drift, invalid ranges, and cross-field consistency.
- Build monthly DQ dashboard with ownership and SLA by source system.


## 5) Portfolio Optimization Under 50% Funding Constraint

In [11]:
with open(DATA / 'portfolio_optimization_summary.json', encoding='utf-8') as f:
    po = json.load(f)

base = po['baseline_50pct_pro_rata']
opt = po['optimized_50pct']
imp = po['improvement_vs_baseline']

print('Baseline expected profit: {:,.0f}'.format(base['exp_profit']))
print('Optimized expected profit: {:,.0f}'.format(opt['exp_profit']))
print('Profit change: {:+.2f}%'.format(imp['exp_profit_change_pct'] * 100))
print('ECL change: {:+.2f}%'.format(imp['ecl_change_pct'] * 100))
print('Profit/Risk change: {:+.2f}%'.format(imp['profit_to_risk_change_pct'] * 100))


Baseline expected profit: 4,503,927,727
Optimized expected profit: 7,004,906,850
Profit change: +55.53%
ECL change: +40.38%
Profit/Risk change: +10.79%


In [12]:
fig = go.Figure()
fig.add_trace(go.Bar(name='Expected Profit', x=['Baseline', 'Optimized'], y=[base['exp_profit'], opt['exp_profit']]))
fig.add_trace(go.Bar(name='Expected Credit Loss', x=['Baseline', 'Optimized'], y=[base['ecl'], opt['ecl']]))
fig.update_layout(barmode='group', template='plotly_white', title='Portfolio Optimization: Profit vs Risk (50% Capital)')
fig.show()


In [13]:
pd_rows = []
with open(DATA / 'pd_cap_scenarios.csv', newline='', encoding='utf-8') as f:
    r = csv.DictReader(f)
    for row in r:
        pd_rows.append({
            'pd_cap': float(row['pd_cap']),
            'profit_change_pct': float(row['exp_profit_change_pct_vs_baseline']) * 100,
            'ecl_change_pct': float(row['ecl_change_pct_vs_baseline']) * 100,
        })

fig = go.Figure()
fig.add_trace(go.Scatter(x=[x['pd_cap'] for x in pd_rows], y=[x['profit_change_pct'] for x in pd_rows], mode='lines+markers', name='Profit Change (%)'))
fig.add_trace(go.Scatter(x=[x['pd_cap'] for x in pd_rows], y=[x['ecl_change_pct'] for x in pd_rows], mode='lines+markers', name='ECL Change (%)'))
fig.update_layout(template='plotly_white', title='PD Cap Trade-off vs Baseline', xaxis_title='PD Cap', yaxis_title='Change (%)')
fig.show()


## 6) Churn Analysis + CLV Priority

In [14]:
with open(DATA / 'churn_profile_summary.json', encoding='utf-8') as f:
    cp = json.load(f)
with open(DATA / 'churn_model_metrics.json', encoding='utf-8') as f:
    cm = json.load(f)
with open(DATA / 'clv_priority_summary.json', encoding='utf-8') as f:
    clv = json.load(f)

selected = cm['selected_model']
sel_metrics = cm['metrics'][selected]['recall_optimized']

print('Overall churn rate: {:.2f}%'.format(cp['overall_churn_rate'] * 100))
print('Selected churn model:', selected)
print('Selected model AUC: {:.4f}'.format(sel_metrics['auc']))
print('Selected model Recall_churn: {:.4f}'.format(sel_metrics['recall_churn']))
print('Top CLV priority size:', clv['top_n'])
print('Top CLV avg churn probability: {:.2f}%'.format(clv['avg_churn_probability_top_n'] * 100))


Overall churn rate: 57.79%
Selected churn model: random_forest
Selected model AUC: 0.9047
Selected model Recall_churn: 0.9897
Top CLV priority size: 5000
Top CLV avg churn probability: 67.17%


In [15]:
grade_seg = []
with open(DATA / 'segment_churn_grade.csv', newline='', encoding='utf-8') as f:
    r = csv.DictReader(f)
    for row in r:
        grade_seg.append((row['segment'], float(row['churn_rate']) * 100))

fig = go.Figure()
fig.add_trace(go.Bar(x=[x[0] for x in grade_seg], y=[x[1] for x in grade_seg], name='Churn Rate (%)'))
fig.update_layout(template='plotly_white', title='Churn Rate by Grade', xaxis_title='Grade', yaxis_title='Churn Rate (%)')
fig.show()


In [16]:
top = []
with open(DATA / 'clv_retention_priority_top.csv', newline='', encoding='utf-8') as f:
    r = csv.DictReader(f)
    for row in r:
        top.append({
            'rank': int(row['rank']),
            'churn_probability': float(row['churn_probability']),
            'clv_estimate': float(row['clv_estimate']),
            'priority': float(row['retention_priority_score'])
        })

top.sort(key=lambda x: x['rank'])
N = len(top)
step = max(1, N // 10)

deciles = []
for i in range(10):
    start = i * step
    end = N if i == 9 else min(N, (i + 1) * step)
    chunk = top[start:end]
    if not chunk:
        continue
    deciles.append({
        'decile': i + 1,
        'avg_prob': mean([x['churn_probability'] for x in chunk]) * 100,
        'avg_clv_k': mean([x['clv_estimate'] for x in chunk]) / 1000,
        'avg_priority_k': mean([x['priority'] for x in chunk]) / 1000,
    })

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08)
fig.add_trace(go.Scatter(x=[x['decile'] for x in deciles], y=[x['avg_prob'] for x in deciles], mode='lines+markers', name='Avg Churn Probability (%)'), row=1, col=1)
fig.add_trace(go.Scatter(x=[x['decile'] for x in deciles], y=[x['avg_clv_k'] for x in deciles], mode='lines+markers', name='Avg CLV (k)'), row=2, col=1)
fig.add_trace(go.Scatter(x=[x['decile'] for x in deciles], y=[x['avg_priority_k'] for x in deciles], mode='lines+markers', name='Avg Priority Score (k)'), row=3, col=1)
fig.update_layout(template='plotly_white', height=900, title='CLV-Based Retention Priority Deciles (D1 = Highest Priority)')
fig.update_xaxes(title_text='Priority Decile', row=3, col=1)
fig.show()


**Retention strategy (actionable):**
- Prioritize customers with both high churn probability and high CLV score.
- Separate campaigns for voluntary exits (re-lending offer) vs risk exits (early intervention, restructuring).
- Track monthly KPI loop: retention conversion, risk migration, and intervention ROI.


## 7) Full-Rerun Command (Already Executed on Full Dataset)

```bash
cd 5107
python3 scripts/run_full_pipeline.py --input_csv /Users/zhongxinxin/Desktop/nusterm2/DA5107/Assignment_data.csv --clean_work_dir
```

This notebook now keeps the key analysis logic and result displays in one place for report delivery.
